# N3: Optimized LSTM with Class Weights + Learning Rate Scheduling

Enhanced LSTM models with optimization techniques to improve AUC:
1. **Class Weights** - Handle 5:1 class imbalance (Real class weighted 5x heavier)
2. **Fixed Epochs** - Train for 30 epochs
3. **Learning Rate Scheduling** - Reduce LR when validation loss plateaus

Expected AUC improvement: 0.9588 → 0.965+ (0.2-0.3% gain)

## 1. Import Required Libraries and Load Data

In [ ]:
import numpy as np
import pandas as pd
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import gc
import time
from tqdm import tqdm
from torch.utils.data import TensorDataset, DataLoader
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score
from transformers import AutoTokenizer, AutoModel

# Setup device for PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("✅ All imports successful")
print(f"   PyTorch Device: {device}")
print(f"   Using LSTM with Class Weights + Learning Rate Scheduling")

✅ All imports successful
   PyTorch Device: cuda
   Using LSTM with Class Weights + Early Stopping + LR Scheduling


## 2. Load Data

In [2]:
train_path = '../../../data/splited/train_set.csv'
val_path = '../../../data/splited/val_set.csv'

df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)

y_train = df_train['label'].values
y_val = df_val['label'].values

print(f"✅ Data loaded: Train {df_train.shape} | Val {df_val.shape}")
print(f"   Labels: Train {(y_train==0).sum()} fake / {(y_train==1).sum()} real")
print(f"           Val {(y_val==0).sum()} fake / {(y_val==1).sum()} real")

✅ Data loaded: Train (3788, 28) | Val (474, 28)
   Labels: Train 3143 fake / 645 real
           Val 393 fake / 81 real


## 3. Generate TF-IDF Embeddings (900-dim)

In [3]:
texts_train_strict = df_train['text_strict'].fillna('').tolist()
texts_val_strict = df_val['text_strict'].fillna('').tolist()

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_df=0.95, min_df=2, sublinear_tf=True)
X_train_tfidf_full = tfidf.fit_transform(texts_train_strict)
X_val_tfidf_full = tfidf.transform(texts_val_strict)

svd = TruncatedSVD(n_components=900, random_state=42)
X_train_tfidf = svd.fit_transform(X_train_tfidf_full)
X_val_tfidf = svd.transform(X_val_tfidf_full)

print(f"✅ TF-IDF embeddings: Train {X_train_tfidf.shape} | Val {X_val_tfidf.shape}")

✅ TF-IDF embeddings: Train (3788, 900) | Val (474, 900)


## 4. Extract PhoBERT Embeddings (Layer 12 Only)

In [4]:
texts_train_loose = df_train['text_loose'].fillna('').tolist()
texts_val_loose = df_val['text_loose'].fillna('').tolist()

# ✅ Load PhoBERT tokenizer and model ONCE (Pretrained model: vinai/phobert-base-v2)
print("📦 Loading PRETRAINED PhoBERT model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base-v2')
phobert_model = AutoModel.from_pretrained('vinai/phobert-base-v2',
                                           output_hidden_states=True).to(device).eval()
print("✅ PhoBERT (Pretrained) loaded successfully")

def extract_layer_embeddings(texts, tokenizer, model, layer_num=12, batch_size=16):
    """Extract PhoBERT embeddings from a specific layer (tokenizer + model passed in)"""
    embeddings = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=f"Layer {layer_num}", leave=False):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=256)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs, output_hidden_states=True)
            
            # Get embeddings from specified layer
            hidden_states = outputs.hidden_states[layer_num]  # (batch, seq, hidden)
            
            # Use [CLS] token (first token)
            cls_tokens = hidden_states[:, 0, :].cpu().numpy()
            embeddings.extend(cls_tokens)
            del outputs, inputs
            torch.cuda.empty_cache()
    
    return np.array(embeddings)

# Extract Layer 12 ONLY (best performing layer)
print("\n📊 Extracting PhoBERT Layer 12 (after 12 encoders)...")
X_train_phobert_l12 = extract_layer_embeddings(texts_train_loose, tokenizer, phobert_model, layer_num=12, batch_size=16)
X_val_phobert_l12 = extract_layer_embeddings(texts_val_loose, tokenizer, phobert_model, layer_num=12, batch_size=16)
print(f"✅ PhoBERT Layer 12 embeddings: Train {X_train_phobert_l12.shape} | Val {X_val_phobert_l12.shape}")

# Clean up PhoBERT model to free memory
phobert_model.to('cpu')
del phobert_model, tokenizer
gc.collect()
print("✅ PhoBERT model released from GPU")

📦 Loading PRETRAINED PhoBERT model and tokenizer...


Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ PhoBERT (Pretrained) loaded successfully

📊 Extracting PhoBERT Layer 12 (after 12 encoders)...


✅ PhoBERT Layer 12 embeddings: Train (3788, 768) | Val (474, 768)
✅ PhoBERT model released from GPU


## 5. Extract Hand-Crafted Features & Calculate Class Weights

In [5]:
exclude_cols = {'id', 'label', 'text_strict', 'text_loose', 'post_message'}
numeric_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [col for col in numeric_cols 
                if col not in exclude_cols and col.strip() != '' and not col.startswith('Unnamed')]

if feature_cols:
    X_train_features = df_train[feature_cols].fillna(0).values
    X_val_features = df_val[feature_cols].fillna(0).values
else:
    X_train_features = np.array([]).reshape(len(df_train), 0)
    X_val_features = np.array([]).reshape(len(df_val), 0)

# Normalize hand-crafted features
scaler_features = StandardScaler()
X_train_features_scaled = scaler_features.fit_transform(X_train_features) if X_train_features.shape[1] > 0 else X_train_features
X_val_features_scaled = scaler_features.transform(X_val_features) if X_val_features.shape[1] > 0 else X_val_features

print(f"✅ Hand-crafted features: {len(feature_cols)} features")
print(f"   Train {X_train_features_scaled.shape} | Val {X_val_features_scaled.shape}")

# ⭐ OPTIMIZATION 1: Calculate Class Weights for imbalanced data
print("\n" + "="*80)
print("OPTIMIZATION 1: CLASS WEIGHTS FOR IMBALANCED DATA")
print("="*80)
class_counts = np.bincount(y_train)
total_samples = len(y_train)
class_weights = total_samples / (len(class_counts) * class_counts)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

print(f"\n📊 Class Distribution:")
print(f"   Fake (0): {class_counts[0]:,} samples → weight: {class_weights[0]:.4f}")
print(f"   Real (1): {class_counts[1]:,} samples → weight: {class_weights[1]:.4f}")
print(f"   Weight Ratio: {class_weights[1] / class_weights[0]:.2f}x")
print(f"   ✅ Real class will be penalized {class_weights[1]/class_weights[0]:.1f}x heavier when misclassified")

✅ Hand-crafted features: 23 features
   Train (3788, 23) | Val (474, 23)

OPTIMIZATION 1: CLASS WEIGHTS FOR IMBALANCED DATA

📊 Class Distribution:
   Fake (0): 3,143 samples → weight: 0.6026
   Real (1): 645 samples → weight: 2.9364
   Weight Ratio: 4.87x
   ✅ Real class will be penalized 4.9x heavier when misclassified


## 6. Define LSTM Architecture

In [21]:
class LSTMNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.5):
        super(LSTMNet, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2)
        )
    
    def forward(self, x):
        # x shape: (batch, seq_len=1, input_dim)
        lstm_out, (h_n, c_n) = self.lstm(x)  # h_n: (num_layers, batch, hidden_dim)
        # Use last hidden state from top layer
        last_hidden = h_n[-1]  # (batch, hidden_dim)
        out = self.fc(last_hidden)  # (batch, 2)
        return out

print("✅ LSTM architecture defined")
print("\n" + "="*80)
print("OPTIMIZATION SUMMARY")
print("="*80)
print("✅ OPTIMIZATION 1: Class Weights - Real class weighted 5x heavier")
print("✅ OPTIMIZATION 2: Fixed Epochs - Train for 30 epochs")
print("✅ OPTIMIZATION 3: Learning Rate Scheduling - Reduce LR when stuck")

✅ LSTM architecture defined

OPTIMIZATION SUMMARY
✅ OPTIMIZATION 1: Class Weights - Real class weighted 5x heavier
✅ OPTIMIZATION 2: Fixed Epochs - Train for 30 epochs
✅ OPTIMIZATION 3: Learning Rate Scheduling - Reduce LR when stuck


## 7. Model 1: Optimized LSTM with TF-IDF + PhoBERT Layer 12

In [22]:
# Combine: TF-IDF + PhoBERT Layer 12
X_train_model1 = np.hstack([X_train_tfidf, X_train_phobert_l12])
X_val_model1 = np.hstack([X_val_tfidf, X_val_phobert_l12])

print(f"Model 1 input (TF + Layer 12): Train={X_train_model1.shape}, Val={X_val_model1.shape}")

# Create tensors and DataLoaders
train_dataset_m1 = TensorDataset(torch.from_numpy(X_train_model1).float().unsqueeze(1),
                                 torch.from_numpy(y_train).long())
val_dataset_m1 = TensorDataset(torch.from_numpy(X_val_model1).float().unsqueeze(1),
                               torch.from_numpy(y_val).long())

train_loader_m1 = DataLoader(train_dataset_m1, batch_size=32, shuffle=True)
val_loader_m1 = DataLoader(val_dataset_m1, batch_size=32, shuffle=False)

# Initialize and train LSTM Model 1
model1 = LSTMNet(input_dim=X_train_model1.shape[1]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model1.parameters(), lr=0.001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=False)

print("🔄 Training Model 1 (TF + PhoBERT Layer 12)...")
start_time = time.time()

for epoch in range(30):
    # Training
    model1.train()
    train_loss = 0
    for X_batch, y_batch in train_loader_m1:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model1(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    # Validation
    model1.eval()
    val_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader_m1:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model1(X_batch)
            loss = criterion(pred, y_batch)
            val_loss += loss.item()
            all_preds.extend(pred.argmax(1).cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    
    val_loss_avg = val_loss / len(val_loader_m1)
    val_acc = accuracy_score(all_labels, all_preds)
    
    scheduler.step(val_loss_avg)
    
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1:2d}: Train Loss={train_loss/len(train_loader_m1):.4f}, "
              f"Val Loss={val_loss_avg:.4f}, Val Acc={val_acc:.4f}")

train_time_m1 = time.time() - start_time
print(f"✅ Model 1 training completed in {train_time_m1:.2f}s\n")

Model 1 input (TF + Layer 12): Train=(3788, 1668), Val=(474, 1668)
🔄 Training Model 1 (TF + PhoBERT Layer 12)...
  Epoch  5: Train Loss=0.2787, Val Loss=0.3273, Val Acc=0.8987
  Epoch 10: Train Loss=0.1782, Val Loss=0.4021, Val Acc=0.8966
  Epoch 15: Train Loss=0.1246, Val Loss=0.4124, Val Acc=0.8840
  Epoch 20: Train Loss=0.0990, Val Loss=0.4766, Val Acc=0.8924
  Epoch 25: Train Loss=0.0902, Val Loss=0.5169, Val Acc=0.9051
  Epoch 30: Train Loss=0.0937, Val Loss=0.5018, Val Acc=0.8945
✅ Model 1 training completed in 10.86s



### 7.1 Evaluate Model 1

In [23]:
print("\n" + "="*80)
print("MODEL 1: EVALUATION (TF-IDF + PhoBERT Layer 12)")
print("="*80)

# Evaluation
model1.eval()
with torch.no_grad():
    X_val_m1_tensor = torch.from_numpy(X_val_model1).float().unsqueeze(1).to(device)
    outputs_val = model1(X_val_m1_tensor)
    y_val_pred_m1 = outputs_val.argmax(dim=1).cpu().numpy()
    y_val_proba_m1 = torch.softmax(outputs_val, dim=1)[:, 1].cpu().numpy()

# Calculate metrics
f1_m1 = f1_score(y_val, y_val_pred_m1, average='weighted')
auc_m1 = roc_auc_score(y_val, y_val_proba_m1)
acc_m1 = accuracy_score(y_val, y_val_pred_m1)
prec_m1 = precision_score(y_val, y_val_pred_m1, average='weighted')
rec_m1 = recall_score(y_val, y_val_pred_m1, average='weighted')

print(f"\n✅ Model 1 Metrics:")
print(f"   F1 Score:  {f1_m1:.4f}")
print(f"   AUC Score: {auc_m1:.4f}  ⭐")
print(f"   Accuracy:  {acc_m1:.4f}")
print(f"   Precision: {prec_m1:.4f}")
print(f"   Recall:    {rec_m1:.4f}")

results_model1 = {
    'F1': f1_m1,
    'AUC': auc_m1,
    'Accuracy': acc_m1,
    'Precision': prec_m1,
    'Recall': rec_m1
}


MODEL 1: EVALUATION (TF-IDF + PhoBERT Layer 12)

✅ Model 1 Metrics:
   F1 Score:  0.8978
   AUC Score: 0.9468  ⭐
   Accuracy:  0.8945
   Precision: 0.9031
   Recall:    0.8945


## 8. Model 1.1: Optimized LSTM with TF-IDF + Layer 12 + Hand-Crafted Features

In [24]:
# Combine: TF-IDF + PhoBERT Layer 12 + Hand-crafted
X_train_model1_1 = np.hstack([X_train_tfidf, X_train_phobert_l12, X_train_features_scaled])
X_val_model1_1 = np.hstack([X_val_tfidf, X_val_phobert_l12, X_val_features_scaled])

print(f"Model 1.1 input (TF + Layer 12 + HC): Train={X_train_model1_1.shape}, Val={X_val_model1_1.shape}")

# Create tensors and DataLoaders
train_dataset_m1_1 = TensorDataset(torch.from_numpy(X_train_model1_1).float().unsqueeze(1),
                                   torch.from_numpy(y_train).long())
val_dataset_m1_1 = TensorDataset(torch.from_numpy(X_val_model1_1).float().unsqueeze(1),
                                 torch.from_numpy(y_val).long())

train_loader_m1_1 = DataLoader(train_dataset_m1_1, batch_size=32, shuffle=True)
val_loader_m1_1 = DataLoader(val_dataset_m1_1, batch_size=32, shuffle=False)

# Initialize and train LSTM Model 1.1
model1_1 = LSTMNet(input_dim=X_train_model1_1.shape[1]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model1_1.parameters(), lr=0.001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=False)

print("🔄 Training Model 1.1 (TF + PhoBERT Layer 12 + Hand-Crafted)...")
start_time = time.time()

for epoch in range(30):
    # Training
    model1_1.train()
    train_loss = 0
    for X_batch, y_batch in train_loader_m1_1:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model1_1(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    # Validation
    model1_1.eval()
    val_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader_m1_1:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model1_1(X_batch)
            loss = criterion(pred, y_batch)
            val_loss += loss.item()
            all_preds.extend(pred.argmax(1).cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    
    val_loss_avg = val_loss / len(val_loader_m1_1)
    val_acc = accuracy_score(all_labels, all_preds)
    
    scheduler.step(val_loss_avg)
    
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1:2d}: Train Loss={train_loss/len(train_loader_m1_1):.4f}, "
              f"Val Loss={val_loss_avg:.4f}, Val Acc={val_acc:.4f}")

train_time_m1_1 = time.time() - start_time
print(f"✅ Model 1.1 training completed in {train_time_m1_1:.2f}s\n")

Model 1.1 input (TF + Layer 12 + HC): Train=(3788, 1691), Val=(474, 1691)
🔄 Training Model 1.1 (TF + PhoBERT Layer 12 + Hand-Crafted)...
  Epoch  5: Train Loss=0.2329, Val Loss=0.2743, Val Acc=0.8945
  Epoch 10: Train Loss=0.1516, Val Loss=0.3261, Val Acc=0.9008
  Epoch 15: Train Loss=0.0867, Val Loss=0.3900, Val Acc=0.9072
  Epoch 20: Train Loss=0.0742, Val Loss=0.4301, Val Acc=0.9051
  Epoch 25: Train Loss=0.0681, Val Loss=0.4238, Val Acc=0.9072
  Epoch 30: Train Loss=0.0662, Val Loss=0.4477, Val Acc=0.9051
✅ Model 1.1 training completed in 10.38s



### 8.1 Evaluate Model 1.1

In [25]:
print("\n" + "="*80)
print("MODEL 1.1: EVALUATION (TF-IDF + PhoBERT Layer 12 + Hand-Crafted)")
print("="*80)

# Evaluation
model1_1.eval()
with torch.no_grad():
    X_val_m1_1_tensor = torch.from_numpy(X_val_model1_1).float().unsqueeze(1).to(device)
    outputs_val = model1_1(X_val_m1_1_tensor)
    y_val_pred_m1_1 = outputs_val.argmax(dim=1).cpu().numpy()
    y_val_proba_m1_1 = torch.softmax(outputs_val, dim=1)[:, 1].cpu().numpy()

# Calculate metrics
f1_m1_1 = f1_score(y_val, y_val_pred_m1_1, average='weighted')
auc_m1_1 = roc_auc_score(y_val, y_val_proba_m1_1)
acc_m1_1 = accuracy_score(y_val, y_val_pred_m1_1)
prec_m1_1 = precision_score(y_val, y_val_pred_m1_1, average='weighted')
rec_m1_1 = recall_score(y_val, y_val_pred_m1_1, average='weighted')

print(f"\n✅ Model 1.1 Metrics:")
print(f"   F1 Score:  {f1_m1_1:.4f}")
print(f"   AUC Score: {auc_m1_1:.4f}  ⭐")
print(f"   Accuracy:  {acc_m1_1:.4f}")
print(f"   Precision: {prec_m1_1:.4f}")
print(f"   Recall:    {rec_m1_1:.4f}")

results_model1_1 = {
    'F1': f1_m1_1,
    'AUC': auc_m1_1,
    'Accuracy': acc_m1_1,
    'Precision': prec_m1_1,
    'Recall': rec_m1_1
}


MODEL 1.1: EVALUATION (TF-IDF + PhoBERT Layer 12 + Hand-Crafted)

✅ Model 1.1 Metrics:
   F1 Score:  0.9070
   AUC Score: 0.9549  ⭐
   Accuracy:  0.9051
   Precision: 0.9099
   Recall:    0.9051


## 9. Final Results: Optimized LSTM Models

In [26]:
print("\n" + "="*80)
print("FINAL RESULTS: OPTIMIZED LSTM MODELS (Class Weights + LR Scheduling - 30 Fixed Epochs)")
print("="*80)

# Create comparison DataFrame
results_data = {
    'Model': [
        'Model 1 (TF + PhoBERT L12)',
        'Model 1.1 (TF + PhoBERT L12 + HC)'
    ],
    'F1 Score': [
        f"{results_model1['F1']:.4f}",
        f"{results_model1_1['F1']:.4f}"
    ],
    'AUC Score': [
        f"{results_model1['AUC']:.4f}",
        f"{results_model1_1['AUC']:.4f}"
    ],
    'Accuracy': [
        f"{results_model1['Accuracy']:.4f}",
        f"{results_model1_1['Accuracy']:.4f}"
    ],
    'Precision': [
        f"{results_model1['Precision']:.4f}",
        f"{results_model1_1['Precision']:.4f}"
    ],
    'Recall': [
        f"{results_model1['Recall']:.4f}",
        f"{results_model1_1['Recall']:.4f}"
    ]
}

results_df = pd.DataFrame(results_data)
print("\n" + "─"*120)
print("OPTIMIZED RESULTS (Class Weights + LR Scheduling - 30 Fixed Epochs)")
print("─"*120 + "\n")
print(results_df.to_string(index=False))

# Best models
auc_scores = [results_model1['AUC'], results_model1_1['AUC']]
best_auc_idx = np.argmax(auc_scores)
best_model = results_data['Model'][best_auc_idx]
best_auc = auc_scores[best_auc_idx]

print(f"\n" + "="*80)
print("🏆 BEST OPTIMIZED MODEL")
print("="*80)
print(f"\n✅ Best Model: {best_model}")
print(f"   AUC Score: {best_auc:.4f}")
if best_auc_idx == 0:
    print(f"   F1: {results_model1['F1']:.4f} | Accuracy: {results_model1['Accuracy']:.4f}")
else:
    print(f"   F1: {results_model1_1['F1']:.4f} | Accuracy: {results_model1_1['Accuracy']:.4f}")

print(f"\n" + "="*80)
print("IMPROVEMENT OVER BASELINE (n1_lstm.ipynb: 0.9588)")
print("="*80)
improvement = (best_auc - 0.9588) * 100
print(f"\n{best_auc:.4f} vs 0.9588 = +{improvement:+.2f}% improvement ✅")
if improvement >= 0.2:
    print(f"\n🎯 Target achieved! (Expected 0.2-0.3% improvement)")
else:
    print(f"\n❌ Improvement is less than expected (0.2-0.3%)")

print(f"\n✅ Optimization training complete!")


FINAL RESULTS: OPTIMIZED LSTM MODELS (Class Weights + LR Scheduling - 30 Fixed Epochs)

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
OPTIMIZED RESULTS (Class Weights + LR Scheduling - 30 Fixed Epochs)
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

                            Model F1 Score AUC Score Accuracy Precision Recall
       Model 1 (TF + PhoBERT L12)   0.8978    0.9468   0.8945    0.9031 0.8945
Model 1.1 (TF + PhoBERT L12 + HC)   0.9070    0.9549   0.9051    0.9099 0.9051

🏆 BEST OPTIMIZED MODEL

✅ Best Model: Model 1.1 (TF + PhoBERT L12 + HC)
   AUC Score: 0.9549
   F1: 0.9070 | Accuracy: 0.9051

IMPROVEMENT OVER BASELINE (n1_lstm.ipynb: 0.9588)

0.9549 vs 0.9588 = +-0.39% improvement ✅

❌ Improvement is less than expected (0.2-0.3%)

✅ Optimization training complete!
